In [3]:
import os
import re
import pdfplumber
import spacy
import numpy as np
import faiss
import requests
import json
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModel
from sklearn.cluster import KMeans
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics.pairwise import cosine_similarity
import string

# Load Spacy model for Named Entity Recognition
nlp = spacy.load('en_core_web_sm')

# Load Legal-BERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
bert_model = TFAutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")

# 41 Legal categories
LEGAL_CATEGORIES = [
    "Agreement Date", "Parties", "Effective Date", "Expiration Date", "Governing Law",
    "Non-Compete", "Exclusivity", "Termination for Convenience", "Revenue/Profit Sharing",
    "Indemnification", "Limitation of Liability", "Confidentiality", "Dispute Resolution",
    "Notice Period", "Force Majeure", "Intellectual Property", "Governing Language", 
    "Jurisdiction", "Severability", "Waiver", "Assignment", "Amendments", "Entire Agreement", 
    "Payment Terms", "Warranty", "Guarantees", "Obligations", "Representations and Warranties", 
    "Rights and Remedies", "Breach of Contract", "Third Party Beneficiaries", "Insurance",
    "Termination Clause", "Automatic Renewal", "Modification of Terms", "Service Level Agreement",
    "Data Protection", "Employee Liability", "Environmental Impact", "Arbitration", 
    "Mediation", "Statutory Requirements"
]

# Category embedding cache (to avoid recalculating)
category_embeddings = {}

# Function for text normalization
def text_normalization(text):
    text = text.lower().translate(str.maketrans("", "", re.escape(string.punctuation)))
    return re.sub(r'\s+', ' ', text).strip()

# Function for Named Entity Recognition (NER) using Spacy
def named_entity_recognition(text):
    doc = nlp(text)
    return [(entity.text, entity.label_) for entity in doc.ents]

# Generate embedding for a given text using Legal-BERT
def get_embedding_for_text(text):
    inputs = tokenizer(text, return_tensors="tf", padding=True, truncation=True)
    outputs = bert_model(inputs['input_ids'])
    return tf.reduce_mean(outputs.last_hidden_state, axis=1).numpy()

# Function to calculate cosine similarity
def calculate_similarity(embedding1, embedding2):
    return cosine_similarity(embedding1, embedding2)[0][0]

# Categorize sentence by finding the closest matching legal category using Legal-BERT embeddings
def categorize_sentence(sentence):
    sentence_embedding = get_embedding_for_text(sentence)
    matched_categories = []
    for category in LEGAL_CATEGORIES:
        if category not in category_embeddings:
            category_embeddings[category] = get_embedding_for_text(category)
        similarity = calculate_similarity(sentence_embedding, category_embeddings[category])
        if similarity > 0.7:  # Set similarity threshold
            matched_categories.append(category)
    return matched_categories

# NLP-based chunking: splits text and categorizes chunks based on closest legal categories
def nlp_based_chunking(text):
    sentences = re.split(r'(?<=[.!?]) +', text)  # Split text into sentences
    chunks = []
    for sentence in sentences:
        matched_categories = categorize_sentence(sentence)
        if matched_categories:
            for category in matched_categories:
                chunks.append({"category": category, "text": sentence.strip()})
        else:
            chunks.append({"category": "General", "text": sentence.strip()})  # Fallback to 'General'
    return chunks

# Function to extract content and metadata from PDF files with NLP-based chunking
def process_file(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = "\n".join([page.extract_text() for page in pdf.pages if page.extract_text()])
            normalized_text = text_normalization(full_text)
            named_entities = named_entity_recognition(normalized_text)
            
            # NLP-based chunking for both category and general chunks
            chunks = nlp_based_chunking(normalized_text)
            
            return {
                "file_name": os.path.basename(pdf_path),
                "normalized_text": normalized_text,
                "named_entities": named_entities,
                "chunks": chunks,
            }
    except Exception as e:
        print(f"Error processing file {pdf_path}: {e}")
        return None

# Function to preprocess all files in a directory
def process_files(pdf_directory):
    file_paths = [os.path.join(pdf_directory, f) for f in os.listdir(pdf_directory) if f.lower().endswith('.pdf')]
    with ThreadPoolExecutor() as executor:
        preprocessed_docs = [result for result in executor.map(process_file, file_paths) if result]
    
    if not preprocessed_docs:
        raise ValueError("No valid documents were processed.")
    
    return preprocessed_docs

# Function to generate chunk embeddings in batches with error handling
def get_document_embeddings_batch(chunks, batch_size=16):
    embeddings = []
    for i in range(0, len(chunks), batch_size):
        batch_texts = [chunk['text'] for chunk in chunks[i:i + batch_size]]
        if not batch_texts:
            continue
        
        inputs = tokenizer(batch_texts, return_tensors="tf", padding=True, truncation=True)
        outputs = bert_model(inputs['input_ids'])
        batch_embeddings = tf.reduce_mean(outputs.last_hidden_state, axis=1).numpy()
        
        if batch_embeddings.shape[0] > 0:
            embeddings.append(batch_embeddings)
    
    if embeddings:
        return np.vstack(embeddings)
    else:
        raise ValueError("No valid embeddings found in the document batch.")

# Initialize FAISS index
def initialize_faiss_index(embedding_dimension):
    return faiss.IndexFlatL2(embedding_dimension)

# Add embeddings to FAISS
def add_embeddings_to_faiss(faiss_index, embeddings):
    faiss_index.add(embeddings.astype('float32'))

# Function to perform search on FAISS index
def search_faiss(faiss_index, query_embedding, top_k=5):
    query_embedding = np.array(query_embedding, dtype=np.float32)
    distances, indices = faiss_index.search(query_embedding, top_k)
    return distances, indices

# K-Means clustering for document chunks
def prepare_faiss_and_clusters(docs, num_clusters):
    all_chunks = [chunk for doc in docs for chunk in doc['chunks']]  # Flatten all chunks
    chunk_document_mapping = [doc['file_name'] for doc in docs for chunk in doc['chunks']]  # Track which document each chunk belongs to
    embeddings = get_document_embeddings_batch(all_chunks)
    faiss_index = initialize_faiss_index(embeddings.shape[1])
    add_embeddings_to_faiss(faiss_index, embeddings)

    kmeans_model = KMeans(n_clusters=num_clusters, random_state=42)
    clusters = kmeans_model.fit_predict(embeddings)

    return faiss_index, embeddings, clusters, kmeans_model, all_chunks, chunk_document_mapping

# Function to preprocess query and generate embedding
def preprocess_query(query):
    normalized_query = text_normalization(query)
    inputs = tokenizer(normalized_query, return_tensors="tf", padding=True, truncation=True)
    outputs = bert_model(inputs['input_ids'])
    return tf.reduce_mean(outputs.last_hidden_state, axis=1).numpy()

# Abstractive summarization using LLaMA with enhanced prompt engineering for query
def generate_llama_summary_with_prompt(chunk_text, named_entities, matched_categories, query=None):
    combined_text = f"{chunk_text}\n\nNamed Entities: {' '.join([entity[0] for entity in named_entities])}"
    categories_text = ', '.join(matched_categories)

    if query:
        # Make the prompt more specific to extract only relevant parts
        system_message = f"Extract information related to the query '{query}'. Focus on the following categories: {categories_text}. Do not provide irrelevant details."
    else:
        system_message = "Summarize the following legal document."

    data = {
        "model": "llama3.1",
        "prompt": f"{system_message}\n\n{combined_text}\n\nAbstract Summary:"
    }

    response = requests.post("http://127.0.0.1:11434/api/generate", json=data, stream=True)
    final_summary = ""
    
    # Handle streaming response from the LLaMA model
    for line in response.iter_lines():
        if line:
            data = json.loads(line.decode('utf-8'))
            final_summary += data.get("response", "")
            if data.get("done", False):
                break

    return final_summary.strip()

def dynamic_summary_mode(docs, faiss_index, clusters, kmeans_model, all_chunks, chunk_document_mapping, mode="query", query=None, top_k=2):
    if mode == "query" and query:
        print(f"Processing query: {query}")
        query_embedding = preprocess_query(query)

        # Predict the cluster for the query
        query_cluster = kmeans_model.predict(query_embedding)[0]
        print(f"Query belongs to cluster: {query_cluster}")

        # Match query with categories using NLP
        matched_categories = categorize_sentence(query)
        print(f"Matched Categories for Query: {matched_categories}")

        # Get chunks in the predicted cluster
        relevant_indices = [i for i, cluster in enumerate(clusters) if cluster == query_cluster]

        # If no categories are matched, fall back to general search
        specific_category_chunks = [all_chunks[i] for i in relevant_indices if any(cat in matched_categories for cat in all_chunks[i]['category'])]
        if not specific_category_chunks:
            specific_category_chunks = [all_chunks[i] for i in relevant_indices]  # Fallback: Search all chunks if no category match

        if not specific_category_chunks:
            raise ValueError("No chunks found for the matched categories or fallback search.")

        # Preprocess the query and generate embedding
        query_embedding = preprocess_query(query)

        # Search within the relevant chunks' FAISS index
        cluster_embeddings = np.array([get_document_embeddings_batch([specific_category_chunks[i]]) for i in range(len(specific_category_chunks))]).reshape(len(specific_category_chunks), -1)
        cluster_faiss_index = initialize_faiss_index(cluster_embeddings.shape[1])
        add_embeddings_to_faiss(cluster_faiss_index, cluster_embeddings)

        # Perform FAISS search
        distances, cluster_specific_indices = search_faiss(cluster_faiss_index, query_embedding, top_k=top_k)

        # Ensure that we don't go out of bounds
        for rank, i in enumerate(cluster_specific_indices[0]):
            if i >= len(relevant_indices):
                print(f"Skipping out-of-bounds index: {i}")
                continue

            original_index = relevant_indices[i]
            chunk = all_chunks[original_index]
            doc_name = chunk_document_mapping[original_index]  # Retrieve document name from mapping
            doc = next(doc for doc in docs if doc['file_name'] == doc_name)  # Get the document details using doc_name
            print(f"Rank {rank+1}, Document: {doc['file_name']}")

            # Only extract relevant information based on the query and categories
            abstractive_summary = generate_llama_summary_with_prompt(chunk['text'], doc['named_entities'], matched_categories, query=query)
            print(f"Query-based Response: {abstractive_summary}\n")
    elif mode == "full":
        print("Generating full-document summaries...")
        for doc in docs:
            for chunk in doc['chunks']:
                abstractive_summary = generate_llama_summary_with_prompt(chunk['text'], doc['named_entities'], ["General"])
                print(f"Document: {doc['file_name']}\nFull Abstractive Summary: {abstractive_summary}\n")

# Running the system
pdf_directory = "D:\\AI_ML - PG\\Capstone Project - Automated Legal document Segmentation\\files_export\\data\\Documents\\IP"
docs = process_files(pdf_directory)

# Prepare FAISS index and KMeans clusters based on the chunks
num_clusters = 2
faiss_index, all_document_embeddings, clusters, kmeans_model, all_chunks, chunk_document_mapping = prepare_faiss_and_clusters(docs, num_clusters)

# Run a query-based summarization
query = "Details: Can a party terminate the contract without cause (solely by giving a notice and allowing a waiting period to expire)?"
dynamic_summary_mode(docs, faiss_index, clusters, kmeans_model, all_chunks, chunk_document_mapping, mode="query", query=query, top_k=2)


C:\Users\DELL\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some layers from the model checkpoint at nlpaueb/legal-bert-base-uncased were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All th

Processing query: Details: Can a party terminate the contract without cause (solely by giving a notice and allowing a waiting period to expire)?
Query belongs to cluster: 0
Matched Categories for Query: ['Revenue/Profit Sharing']
Rank 1, Document: PREMIERBIOMEDICALINC_05_14_2020-EX-10.2-INTELLECTUAL PROPERTY AGREEMENT.PDF
Query-based Response: This is a non-disclosure agreement (NDA) between Premier Biomedical Inc, Technology Health Inc, and Marv Enterprises LLC. The agreement restricts the parties from disclosing confidential information to third parties.

Key points:

* Confidential information includes trade secrets, inventions, and other proprietary information.
* Parties agree not to disclose or use confidential information for personal gain.
* All confidential information must be returned or destroyed upon request of the disclosing party.
* The agreement is governed by Pennsylvania law and any disputes will be resolved in Mercer County, PA.
* Time is of the essence in complying w